Evan Edelstein
EN.605.645.82.SP26

# Module 7 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

# Unification

This is actually Part I of a two part assignment. In a later module, you'll implement a Forward Planner. In order to do that, however, you need to have a unifier. It is important to note that you *only* need to implement a unifier. Although the module talked about resolution, you do not need to implement anything like "standardizing apart". From the unifier's point of view, that should already have been done.

Unification is simply the *syntactic* balancing of expressions. There are only 3 kinds of expressions: constants, lists and (logic) variables. Constants and lists are only equal to each other if they're exactly the same thing or can be made to be the same thing by *binding* a value to a variable.

It really is that simple...expressions must be literally the same (identical) except if one or the other (or both) has a variable in that "spot".

## S-Expressions

With that out of the way, we need a language with which to express our constants, variables and predicates and that language will be based on s-expressions.

**constants** - There are two types of constants, values and predicates. Values should start with an uppercase letter. Fred is a constant value, so is Barney and Food. Predicates are named using lowercase letters. loves is a predicate and so is hates. This is only a convention. Secret: your code does not need to treat these two types of constants differently.

**variables** - these are named using lowercase letters but always start with a question mark. ?x is a variable and so is ?yum. This is not a convention.

**expressions (lists)** - these use the S-expression syntax a la LISP. (loves Fred Wilma) is an expression as is (friend-of Barney Fred) and (loves ?x ?y).

## Parsing

In [38]:
import tokenize
from io import StringIO
from typing import List, Dict, Tuple

This uses the above libraries to build a Lisp structure based on atoms. It is adapted from [simple iterator parser](http://effbot.org/zone/simple-iterator-parser.htm) (deadlink). The first function is the `atom` function.

In [39]:
def atom(next, token):
    if token[1] == "(":
        out = []
        token = next()
        while token[1] != ")":
            out.append(atom(next, token))
            token = next()
            if token[1] == " ":
                token = next()
        return out
    elif token[1] == "?":
        token = next()
        return "?" + token[1]
    else:
        return token[1]

The next function is the actual `parse` function:

In [40]:
def parse(exp):
    src = StringIO(exp).readline
    tokens = tokenize.generate_tokens(src)
    return atom(tokens.__next__, tokens.__next__())

**Note** there was a change between 2.7 and 3.0 that "hid" the next() function in the tokenizer.

From a Python perspective, we want to turn something like "(loves Fred ?x)" to ["loves" "Fred" "?x"] and then work with the second representation as a list of strings. The strings then have the syntactic meaning we gave them previously.

In [41]:
parse("Fred")

'Fred'

In [42]:
parse("?x")

'?x'

In [43]:
parse("(loves Fred ?x)")

['loves', 'Fred', '?x']

In [44]:
parse("(father_of Barney (son_of Barney))")

['father_of', 'Barney', ['son_of', 'Barney']]

## Unifier

Now that that's out of the way, here is the imperative pseudocode for unification. This is a classic recursive program with a number of base cases. Students for some reason don't like it, try the algorithm in the book, can't get it to work and then come back to this pseudocode.

Work through the algorithm by hand with your Self-Check examples if you need to but I'd suggest sticking with this implementation. It does work.

Here is imperative pseudocode for the algorithm:

```
def unification( exp1, exp2):
    # base cases
    if exp1 and exp2 are constants or the empty list:
        if exp1 = exp2 then return {}
        else return FAIL
    if exp1 is a variable:
        if exp1 occurs in exp2 then return FAIL
        else return {exp1/exp2}
    if exp2 is a variable:
        if exp2 occurs in exp1 then return FAIL
        else return {exp2/exp1}

    # inductive step
    first1 = first element of exp1
    first2 = first element of exp2
    result1 = unification( first1, first2)
    if result1 = FAIL then return FAIL
    apply result1 to rest of exp1 and exp2
    result2 = unification( rest of exp1, rest of exp2)
    if result2 = FAIL then return FAIL
    return composition of result1 and result2
```

`unification` can return...

1. `None` (if unification completely fails)
2. `{}` (the empty substitution list) or 
3. a substitution list that has variables as keys and substituted values as values, like {"?x": "Fred"}. 

Note that the middle case sometimes confuses people..."Sam" unifying with "Sam" is not a failure so you return {} because there were no variables so there were no substitutions. You do not need to further resolve variables. If a variable resolves to an expression that contains a variable, you don't need to do the substition.

If you think of a typical database table, there is a column, row and value. This Tuple is a *relation* and in some uses of unification, the "thing" in the first spot..."love" above is called the relation. If you have a table of users with user_id, username and the value then the relation is:

`(login ?user_id ?username)`

*most* of the time, the relation name is specified. But it's not impossible for the relation name to be represented by a variable:

`(?relation 12345 "smooth_operator")`

Your code should handle this case (the pseudocode does handle this case so all  you have to do is not futz with it).

Our type system is very simple. We can get by with just a few boolean functions. The first tests to see if an expression is a variable.

In [45]:
def is_variable(exp):
    return isinstance(exp, str) and exp[0] == "?"

In [46]:
is_variable("Fred")

False

In [47]:
is_variable("?fred")

True

The second tests to see if an expression is a constant:

In [48]:
def is_constant(exp):
    return isinstance(exp, str) and not is_variable(exp)

In [49]:
is_constant("Fred")

True

In [50]:
is_constant("?fred")

False

In [51]:
is_constant(["loves", "Fred", "?wife"])

False

It might also be useful to know that:

<code>
type( "a")
&lt;type 'str'>
type( "a") == str
True
type( "a") == list
False
type( ["a"]) == list
True
</code>


You need to write the `unification` function described above. It should work with two expressions of the type returned by `parse`. See `unify` for how it will be called. It should return the result of unification for the two expressions as detailed above and in the book. It does not have to make all the necessary substitions (for example, if ?y is bound to ?x and 1 is bound to ?y, ?x doesn't have to be replaced everywhere with 1. It's enough to return {"?x":"?y", "?y":1}. For an actual application, you would need to fix this!)

-----

In [52]:
def is_equal(list_expression1, list_expression2):
    # expressions must be literally the same (identical) except if one or the other (or both) has a variable in that "spot".
    if len(list_expression1) != len(list_expression2):
        return False

    for i, j in zip(list_expression1, list_expression2):
        if is_variable(i) or is_variable(j):
            continue
        if i != j:
            return False
    return True

In [53]:
def occurs(variable, expression):
    return variable in expression

In [54]:
def split_first(exp):
    return exp[0], exp[1:]

In [55]:
def apply(result, list_expression1, list_expression2):
    list_expression1 = [result[i] if is_constant(i) and i in result else i for i in list_expression1]
    list_expression2 = [result[i] if is_constant(i) and i in result else i for i in list_expression2]
    return list_expression1, list_expression2


In [ ]:
def composition(result1: Dict, result2: Dict, trace: bool=False):
    result = result1
    if result2:
        result.update(result2)
    if trace:
        print("result: ", result1, result2, result)
    return result

In [57]:
def assign(variable, expression):
    if isinstance(expression, list):
        expression = "(" + " ".join(expression) + ")"
    return {variable: expression}

In [58]:
# should be "is_unmodifiable"
def is_constant_expression(list_expression1, list_expression2) -> Tuple[bool, Dict | None]:
    if (is_constant(list_expression1) and is_constant(list_expression2)) or (len(list_expression1) == 0 or len(list_expression2) == 0):
        if is_equal(list_expression1, list_expression2):
            return True, {}
        return True, None
    return False, None

In [ ]:
def is_assignable(list_expression1, list_expression2) -> Tuple[bool, None | Dict]:
    if is_variable(list_expression1):
        if list_expression1 in list_expression2:
            return True, None
        return True, assign(list_expression1, list_expression2)

    return False, None

In [60]:
# TODO rename
# TODO add debug statements
def base_cases(list_expression1, list_expression2, trace: bool = False):
    result = is_constant_expression(list_expression1, list_expression2)
    if result[0]:
        return result

    result = is_assignable(list_expression1, list_expression2)
    if result[0]:
        return result

    result = is_assignable(list_expression2, list_expression1)
    if result[0]:
        return result
    return False, None

In [61]:
def split_expressions(list_expression1, list_expression2):
    first1, rest1 = list_expression1[0], list_expression1[1:]
    first2, rest2 = list_expression2[0], list_expression2[1:]
    return first1, rest1, first2, rest2

In [62]:
def is_contradiction(result1, result2, trace: bool = False):
    if result1 != result2 and any(key in result1 for key in result2):
        if trace:
            print("")  # TODO
        return True
    return False

In [63]:
def unification(list_expression1, list_expression2, trace=False) -> None | Dict:
    result = base_cases(list_expression1, list_expression2, trace)
    if result[0]:
        return result[1]

    first1, rest1, first2, rest2 = split_expressions(list_expression1, list_expression2)

    result1 = unification(first1, first2, trace)
    if result1 is None:
        return None

    list_expression1, list_expression2 = apply(result1, list_expression1, list_expression2)

    result2 = unification(rest1, rest2, trace)
    if result2 is None or is_contradiction(result1, result2, trace):
        return None

    return composition(result1, result2, trace)

In [64]:
def list_check(parsed_expression):
    if isinstance(parsed_expression, list):
        return parsed_expression
    return [parsed_expression]

The `unification` pseudocode only takes lists so we have to make sure that we only pass a list.
However, this has the side effect of making "foo" unify with ["foo"], at the start.
That's ok.

In [65]:
def unify(s_expression1, s_expression2):
    list_expression1 = list_check(parse(s_expression1))
    list_expression2 = list_check(parse(s_expression2))
    return unification(list_expression1, list_expression2)

**Note** If you see the error,

```
tokenize.TokenError: ('EOF in multi-line statement', (2, 0))
```
You most likely have unbalanced parentheses in your s-expression.

## Test Cases

Use the expressions from the Self Check as your test cases...

In [66]:
self_check_test_cases = [["(son Barney Barney)", "(daughter Wilma Pebbles)", None]]
for case in self_check_test_cases:
    exp1, exp2, expected = case
    actual = unify(exp1, exp2)
    print(f"actual = {actual}")
    print(f"expected = {expected}")
    print("\n")
    assert expected == actual

actual = None
expected = None




Now add at least **five (5)** additional test cases of your own making, explaining exactly what you are testing. They should not be testing the same things as the self check test cases above.

In [67]:
def run_tests(test_cases):
    for i, case in enumerate(test_cases):
        exp1, exp2, expected, message = case
        actual = unify(exp1, exp2)
        print(f"[{i + 1}] Testing {message}...")
        pre = "✅" if actual == expected else "❌"
        print(f"{pre} actual = {actual} | expected = {expected}")
        # print("\n")
        assert expected == actual

In [68]:
# new_test_cases = [["(son Barney Barney)", "(daughter Wilma Pebbles)", None, "non-equal constants"]]
textbook_test_cases = [
    ["(knowns John ?x)", "(knowns John Jane)", {"?x": "Jane"}, "textbook test case - valid unification of single variable"],
    ["(knowns John ?x)", "(knowns ?y Bill)", {"?x": "Bill", "?y": "John"}, "textbook test case - valid unification of two variables"],
    ["(knowns John ?x)", "(knowns ?y (mother ?y))", {"?y": "John", "?x": "(mother ?y)"}, "textbook test case - valid assignment of a variable to an expression"],
    ["(knowns John ?x)", "(knowns Elizabeth ?x)", None, "textbook test case - invalid unification of two constants to the same variable"],
    ["(knowns John ?x)", "(knowns ?x Elizabeth)", None, "textbook test case - invalid unification of two constants to the same variable is order independent"],
]

run_tests(textbook_test_cases)

[1] Testing textbook test case - valid unification of single variable...
✅ actual = {'?x': 'Jane'} | expected = {'?x': 'Jane'}
[2] Testing textbook test case - valid unification of two variables...
✅ actual = {'?y': 'John', '?x': 'Bill'} | expected = {'?x': 'Bill', '?y': 'John'}
[3] Testing textbook test case - valid assignment of a variable to an expression...
✅ actual = {'?y': 'John', '?x': '(mother ?y)'} | expected = {'?y': 'John', '?x': '(mother ?y)'}
[4] Testing textbook test case - invalid unification of two constants to the same variable...
✅ actual = None | expected = None
[5] Testing textbook test case - invalid unification of two constants to the same variable is order independent...
✅ actual = None | expected = None


In [69]:
self_check_test_cases = [
    ["(Fred)", "(Barney)", None, "self check case - invalid unification of non-equal constant"],
    ["(Pebbles)", "(Pebbles)", {}, "self check case - unification of the same constant returns an empty substitution list"],
    ["(quarry_worker Fred)", "(quarry_worker ?x)", {"?x": "Fred"}, "self check case - valid unification of single variable"],
    ["(son Barney ?x)", "(son ?y Bam_Bam)", {"?y": "Barney", "?x": "Bam_Bam"}, "self check case - valid unification of two variables"],
    ["(married ?x ?y)", "(married Barney Wilma)", {"?x": "Barney", "?y": "Wilma"}, "self check case - valid unification of two variable within the same expression"],
    ["(son Barney ?x)", "(son ?y (son Barney))", {"?y": "Barney", "?x": "(son Barney)"}, "self check case - valid assignment of a variable to an expression"],
    ["(son Barney ?x)", "(son ?y (son ?y))", {"?y": "Barney", "?x": "(son ?y)"}, "self check case - valid assignment of a variable to an expression with replacement"],
    ["(son Barney Bam_Bam)", "(son ?y (son Barney))", None, "self check case -  invalid unification of a constant function to a constant"],
    ["(loves Fred Fred)", "(loves ?x ?x)", {"?x": "Fred"}, "self check case - valid substitution of the same constant to the same variable"],
    ["(loves George Fred)", "(loves ?y ?y)", None, "self check case - invalid substitution of the different constants to the same variable"],
]
run_tests(self_check_test_cases)

[1] Testing self check case - invalid unification of non-equal constant...
✅ actual = None | expected = None
[2] Testing self check case - unification of the same constant returns an empty substitution list...
✅ actual = {} | expected = {}
[3] Testing self check case - valid unification of single variable...
✅ actual = {'?x': 'Fred'} | expected = {'?x': 'Fred'}
[4] Testing self check case - valid unification of two variables...
✅ actual = {'?y': 'Barney', '?x': 'Bam_Bam'} | expected = {'?y': 'Barney', '?x': 'Bam_Bam'}
[5] Testing self check case - valid unification of two variable within the same expression...
✅ actual = {'?x': 'Barney', '?y': 'Wilma'} | expected = {'?x': 'Barney', '?y': 'Wilma'}
[6] Testing self check case - valid assignment of a variable to an expression...
✅ actual = {'?y': 'Barney', '?x': '(son Barney)'} | expected = {'?y': 'Barney', '?x': '(son Barney)'}
[7] Testing self check case - valid assignment of a variable to an expression with replacement...
✅ actual = {'

In [70]:
new_test_cases = [
    # ["(son Barney Barney)", "(daughter Wilma Pebbles)", None, "non-equal constants"],
]

run_tests(new_test_cases)

## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.